# Pipedrive CRM Data - Exploratory Analysis

This notebook explores the Pipedrive CRM source data to understand structure, relationships, and data quality.

In [ ]:
import pandas as pd
import psycopg2
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# connection settings
conn_params = {
    'host': 'localhost',
    'database': 'postgres',
    'user': 'admin',
    'password': 'admin',
    'port': 5432
}

conn = psycopg2.connect(**conn_params)

## 1. Data Overview

First, let's check what tables we have and their basic stats.

In [ ]:
# get table counts
tables = ['stages', 'activity_types', 'activity', 'deal_changes', 'fields', 'users']

for table in tables:
    query = f"SELECT COUNT(*) as count FROM public.{table}"
    df = pd.read_sql(query, conn)
    print(f"{table}: {df['count'].iloc[0]:,} rows")

## 2. Stages Analysis

Understanding the sales funnel stages.

In [ ]:
# stages
stages_df = pd.read_sql("SELECT * FROM public.stages ORDER BY stage_id", conn)
print("Sales Funnel Stages:")
print(stages_df.to_string(index=False))

# visualize stages
plt.figure(figsize=(10, 6))
plt.barh(range(len(stages_df)), stages_df['stage_id'], color='steelblue')
plt.yticks(range(len(stages_df)), stages_df['stage_name'])
plt.xlabel('Stage ID')
plt.title('Sales Funnel Stages')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Activity Types

What types of activities are tracked?

In [ ]:
# activity types
activity_types_df = pd.read_sql("SELECT * FROM public.activity_types", conn)
print("Activity Types:")
print(activity_types_df.to_string(index=False))

# check which are active
print("\nActive vs Inactive:")
print(activity_types_df['active'].value_counts())

## 4. Deal Changes Analysis

This is the key table - tracks all changes to deals over time.

In [ ]:
# deal changes overview
deal_changes_df = pd.read_sql("""
    SELECT 
        changed_field_key,
        COUNT(*) as change_count,
        COUNT(DISTINCT deal_id) as unique_deals
    FROM public.deal_changes
    GROUP BY changed_field_key
    ORDER BY change_count DESC
""", conn)

print("Deal Changes by Field:")
print(deal_changes_df.to_string(index=False))

In [ ]:
# stage transitions - most important for funnel
stage_changes = pd.read_sql("""
    SELECT 
        changed_field_key,
        new_value as stage_id,
        COUNT(*) as transitions
    FROM public.deal_changes
    WHERE changed_field_key = 'stage_id'
    GROUP BY changed_field_key, new_value
    ORDER BY new_value::integer
""", conn)

print("Stage Transitions:")
print(stage_changes.to_string(index=False))

In [ ]:
# time range of deal changes
time_range = pd.read_sql("""
    SELECT 
        MIN(change_time) as earliest_change,
        MAX(change_time) as latest_change,
        COUNT(DISTINCT deal_id) as total_deals
    FROM public.deal_changes
    WHERE changed_field_key = 'stage_id'
""", conn)

print("Deal Changes Time Range:")
print(time_range.to_string(index=False))

## 5. Activities Analysis

Understanding activities linked to deals.

In [ ]:
# activities overview
activities_df = pd.read_sql("""
    SELECT 
        type,
        COUNT(*) as activity_count,
        COUNT(DISTINCT deal_id) as deals_with_activities,
        COUNT(CASE WHEN done THEN 1 END) as completed_count
    FROM public.activity
    GROUP BY type
    ORDER BY activity_count DESC
""", conn)

print("Activities Summary:")
print(activities_df.to_string(index=False))

In [ ]:
# activities by type - focus on Sales Call 1 and 2
sales_calls = pd.read_sql("""
    SELECT 
        a.type,
        at.name,
        COUNT(*) as count,
        COUNT(DISTINCT a.deal_id) as unique_deals
    FROM public.activity a
    LEFT JOIN public.activity_types at ON a.type = at.type
    WHERE at.id IN (1, 2)
    GROUP BY a.type, at.name
""", conn)

print("Sales Call Activities:")
print(sales_calls.to_string(index=False))

## 6. Data Relationships

Understanding how tables connect.

In [ ]:
# check relationships
print("Key Relationships:\n")

# deals in deal_changes vs activities
deals_in_changes = pd.read_sql("SELECT COUNT(DISTINCT deal_id) as count FROM public.deal_changes WHERE changed_field_key = 'stage_id'", conn)
deals_in_activities = pd.read_sql("SELECT COUNT(DISTINCT deal_id) as count FROM public.activity", conn)

print(f"Unique deals in deal_changes (stage transitions): {deals_in_changes['count'].iloc[0]}")
print(f"Unique deals in activities: {deals_in_activities['count'].iloc[0]}")

# overlap
overlap = pd.read_sql("""
    SELECT COUNT(DISTINCT dc.deal_id) as overlap_count
    FROM public.deal_changes dc
    INNER JOIN public.activity a ON dc.deal_id = a.deal_id
    WHERE dc.changed_field_key = 'stage_id'
""", conn)

print(f"Deals with both stage changes and activities: {overlap['overlap_count'].iloc[0]}")

## 7. Data Quality Checks

Checking for data quality issues.

In [ ]:
# null checks
print("Null Value Checks:\n")

# deal_changes
nulls_changes = pd.read_sql("""
    SELECT 
        COUNT(*) FILTER (WHERE deal_id IS NULL) as null_deal_id,
        COUNT(*) FILTER (WHERE change_time IS NULL) as null_change_time,
        COUNT(*) FILTER (WHERE changed_field_key IS NULL) as null_field_key
    FROM public.deal_changes
""", conn)
print("deal_changes nulls:")
print(nulls_changes.to_string(index=False))

# activities
nulls_activities = pd.read_sql("""
    SELECT 
        COUNT(*) FILTER (WHERE deal_id IS NULL) as null_deal_id,
        COUNT(*) FILTER (WHERE due_to IS NULL) as null_due_to
    FROM public.activity
""", conn)
print("\nactivity nulls:")
print(nulls_activities.to_string(index=False))

## 8. Monthly Funnel Distribution

Preview of what the final report will show.

In [ ]:
# sample monthly distribution
monthly_sample = pd.read_sql("""
    SELECT 
        DATE_TRUNC('month', change_time) as month,
        new_value::integer as stage_id,
        COUNT(DISTINCT deal_id) as deal_count
    FROM public.deal_changes
    WHERE changed_field_key = 'stage_id'
        AND new_value ~ '^[0-9]+$'
    GROUP BY month, new_value::integer
    ORDER BY month DESC, stage_id
    LIMIT 20
""", conn)

print("Sample Monthly Funnel Data:")
print(monthly_sample.to_string(index=False))

## 9. Key Insights

### Findings:

1. **Deal Changes is the primary source** for tracking stage transitions
   - Contains historical changes with timestamps
   - Need to use MIN() to find first entry per stage

2. **Activities table** tracks Sales Call 1 and Sales Call 2
   - Activity type IDs: 1 = Sales Call 1, 2 = Sales Call 2
   - Linked to deals via deal_id

3. **Data structure** supports funnel analysis
   - Stages table provides stage names (1-9)
   - Deal changes track progression
   - Activities provide sub-steps (2.1, 3.1)

4. **Data quality** looks good
   - Minimal nulls in key fields
   - Relationships are consistent

5. **Time range** covers multiple months
   - Supports monthly aggregation
   - Historical data available for trend analysis

In [ ]:
conn.close()
print("Analysis complete!")